# E27 Spectral Correlation: Hammerhead Formation vs $B_N$ Wave Power

**Scientific question:** Do ion cyclotron waves drive the formation of hammerhead features in the PSP SPAN-I proton velocity distributions?

**Hypothesis:** Hammerhead features form when protons encounter cyclotron-resonant magnetic fluctuations in a specific frequency band (~12-35 Hz). If true, the *rate* at which hammerheads appear (occurrence rate, a.k.a. "hammogram") should correlate with magnetic wave power in that band — but NOT with broadband magnetic variability.

**Time range:** 2026/03/11 07:10-08:30 UT (Encounter 27 perihelion approach)

**Approach:**
1. Load $B_N$ (RTN normal component) at full ~293 Hz cadence and hammerhead detection events from the E27 CDF archive
2. Compute a spectrogram to see how magnetic fluctuation power varies with frequency and time
3. Pick a target frequency band suspected to drive hammerhead formation (iterate visually)
4. Correlate spectral power in that band with hammerhead parameters
5. Compare against a disjoint control band (frequencies outside our target) to confirm band-specificity
6. Report both raw and effective-n-corrected p-values to stay honest about significance

**Key distinction:** We correlate against TWO different hammerhead quantities:
- `n_ham` — the *physical density* of the hammerhead population when one is detected (cm$^{-3}$). Tests: do waves make hams DENSER?
- `hamogram_30s` — the *occurrence rate*, i.e. count of hammerhead detections in each 30-second bin. Tests: do waves make hams MORE LIKELY TO FORM?

These are physically distinct and may tell different stories.

## Data setup

This notebook uses **hammerhead CDF files** (processed PSP SPAN-I proton VDF parameters from the [HamPy](https://github.com/srijandas07/HamPy) package) and **PSP FIELDS magnetic field data**.

### Hammerhead CDFs (must be placed manually)

Plotbot expects the hammerhead v02 CDFs in this location:

```
plotbot-v1/
  data/
    cdf_files/
      Hamstrings/
        hamstring_2026-03-07_v02.cdf
        hamstring_2026-03-08_v02.cdf
        hamstring_2026-03-09_v02.cdf
        hamstring_2026-03-10_v02.cdf
        hamstring_2026-03-11_v02.cdf
```

For this notebook the source archive lives at `plotbot-v1/Hamstrings_E27/cdf/v02/` (gitignored). If you receive a new archive from Srijan / Jaye, copy the `hamstring_*_v02.cdf` files into `data/cdf_files/Hamstrings/` before running.

Quick copy command (from the repo root):
```bash
mkdir -p data/cdf_files/Hamstrings
cp Hamstrings_E27/cdf/v02/hamstring_2026-03-*.cdf data/cdf_files/Hamstrings/
```

### Magnetic field data (automatic)

`mag_rtn` pulls PSP FIELDS level-2 RTN magnetic field data automatically via plotbot's `get_data()` the first time it's needed. The file is downloaded from the PSP data center and cached locally under `data/psp/fields/`. No manual placement required.

### Verify before running

The cell below will fail at Step 1 if the hammerhead CDFs aren't found. Quick sanity check:
```bash
ls data/cdf_files/Hamstrings/hamstring_2026-03-11_v02.cdf
```
If that prints the file path, you're good to go.


In [ ]:
# Imports + plot defaults
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d
from scipy.stats import pearsonr, spearmanr

import plotbot
from plotbot import ham, mag_rtn
from plotbot.get_data import get_data

# --- Bigger text across every plot in this notebook ---
plt.rcParams.update({
    'font.size':        14,   # base font size
    'axes.titlesize':   16,   # subplot titles
    'axes.labelsize':   15,   # x/y axis labels
    'xtick.labelsize':  13,   # x tick labels
    'ytick.labelsize':  13,   # y tick labels
    'legend.fontsize':  13,
    'figure.titlesize': 17,   # suptitle
})

In [ ]:
# ==========================================================================
# Step 1: Load datasets + binning config
# ==========================================================================
# We correlate:
#   mag_rtn.bn                    - N component of the magnetic field
#   ham.n_ham / ham.n_core        - hammerhead fraction of the proton population
#   ham.Tperp_ham / ham.Tpar_ham  - hammerhead temperature anisotropy
#   hamogram_30s                  - occurrence rate (detections per bin)
#
# 80-minute window around E27 perihelion -- long enough for statistics,
# short enough that PSP's trajectory is roughly stationary.
trange = ['2026/03/11 07:10:00.000', '2026/03/11 08:30:00.000']

# --- Binning configuration ---
# Per-detection ham quantities get binned into fixed-width time windows
# before correlation with wave power -- same treatment we give hamogram.
# Native binning (instead of rolling-mean smoothing) gives clean independent
# samples without autocorrelation games.
use_single_bin = True   # True: every target uses bin_sec (simple, default)
                        # False: each target uses its own override below
bin_sec        = 120    # seconds per bin -- the knob you'll usually tune

# Per-target overrides (only read when use_single_bin=False)
bin_sec_hamogram = 120  # hamogram occurrence rate binning
bin_sec_n_ham    = 120  # n_ham / n_core binning
bin_sec_t_aniso  = 120  # Tperp / Tpar binning

# Resolved bin sizes -- downstream cells read THESE, not the raw config
_bin_hamogram = bin_sec if use_single_bin else bin_sec_hamogram
_bin_n_ham    = bin_sec if use_single_bin else bin_sec_n_ham
_bin_t_aniso  = bin_sec if use_single_bin else bin_sec_t_aniso

# --- Load data ---
get_data(trange, ham.n_ham)
get_data(trange, ham.n_core)     # needed to form the hammerhead fraction n_ham / n_core
get_data(trange, ham.Tperp_ham)  # needed to form the anisotropy Tperp_ham / Tpar_ham
get_data(trange, ham.Tpar_ham)
get_data(trange, ham.vx_inst_core)  # needed to form the parallel drift velocity
get_data(trange, ham.vy_inst_core)
get_data(trange, ham.vz_inst_core)
get_data(trange, ham.vx_inst_ham)
get_data(trange, ham.vy_inst_ham)
get_data(trange, ham.vz_inst_ham)
get_data(trange, ham.Bx_inst)    # instrument-frame B for projecting v onto b-hat
get_data(trange, ham.By_inst)
get_data(trange, ham.Bz_inst)
get_data(trange, mag_rtn.bn)

print('\n' + '='*78)
print('STEP 1: Load data')
print('  Measuring: Pulling B_N (mag, ~293 Hz) and hammerhead detection events for E27 perihelion window')
print('='*78)
print(f'trange: {trange}')
print(f'Ham: {len(ham.datetime_array)} records')
print(f'  {ham.datetime_array[0]} to {ham.datetime_array[-1]}')
print(f'Mag RTN: {len(mag_rtn.datetime_array)} records')
print(f'  {mag_rtn.datetime_array[0]} to {mag_rtn.datetime_array[-1]}')
dt_ns = np.diff(mag_rtn.datetime_array.astype("int64")).mean()
fs = 1e9 / dt_ns
print(f'  Sample rate: {fs:.1f} Hz')

In [ ]:
# ==========================================================================
# Step 2: Spectrogram of B_N
# ==========================================================================
# Turn the magnetic field time series into power-vs-frequency-vs-time so we
# can ask "how much wave power at each frequency at each moment?"
# 10s sliding FFT windows with 50% overlap -> ~0.1 Hz frequency resolution,
# ~5s time cadence. Any NaNs in the signal are interpolated first because
# scipy's FFT propagates them and turns whole windows blank.
bn_data = np.asarray(mag_rtn.bn, dtype=np.float64)
mag_times = mag_rtn.datetime_array

print('\n' + '='*78)
print('STEP 2: Compute B_N spectrogram')
print('  Measuring: Magnetic power spectral density as a function of frequency and time (10s windows)')
print('='*78)

# Interpolate across any NaNs (they poison FFT windows and create blank bands)
nan_mask = np.isnan(bn_data)
n_nans = nan_mask.sum()
if n_nans > 0:
    idx = np.arange(len(bn_data))
    bn_data = np.interp(idx, idx[~nan_mask], bn_data[~nan_mask])
    print(f'Interpolated across {n_nans} NaN samples before spectrogram')

# Spectrogram parameters
nperseg = int(fs * 10)  # 10-second windows
noverlap = nperseg // 2  # 50% overlap

f, t_spec, Sxx = signal.spectrogram(
    bn_data, fs=fs, nperseg=nperseg, noverlap=noverlap,
    detrend='constant', scaling='density'
)

# Vectorized datetime conversion (no Python loop)
t0_ns = mag_times[0].astype('datetime64[ns]').astype(np.int64)
t_spec_ns = t0_ns + (t_spec * 1e9).astype(np.int64)
t_spec_dt = t_spec_ns.astype('datetime64[ns]')

# Pre-compute log PSD once for plotting (avoid recomputing in every cell)
Sxx_db = 10 * np.log10(Sxx + 1e-20)

print(f'Spectrogram shape: {Sxx.shape}')
print(f'Frequency range: {f[1]:.3f} to {f[-1]:.1f} Hz')
print(f'Frequency resolution: {f[1]-f[0]:.4f} Hz')
print(f'Time bins: {len(t_spec)}')

In [ ]:
# ==========================================================================
# Step 3: 4-panel visual overview (and compute the hammogram)
# ==========================================================================
# Eyeball the data before quantifying: do hammerhead-heavy times line up
# with bright patches in the spectrogram? Four stacked panels sharing the
# same time axis: B_N, spectrogram, n_ham (density), hamogram_30s (rate).
from mpl_toolkits.axes_grid1 import make_axes_locatable

ham_times = ham.datetime_array
# Use the hammerhead FRACTION (n_ham / n_core) instead of raw n_ham.
# This normalizes out total proton density fluctuations -- if both populations
# drift with trajectory, the ratio cancels the drift, leaving only the
# physically interesting "fraction of protons in the hammerhead population."
_n_ham_raw   = np.asarray(ham.n_ham,  dtype=np.float64)
_n_core_raw  = np.asarray(ham.n_core, dtype=np.float64)
# Safe division: mask out bins where n_core <= 0 or NaN
_safe_core = np.where((_n_core_raw > 0) & np.isfinite(_n_core_raw), _n_core_raw, np.nan)
n_ham_data = _n_ham_raw / _safe_core     # dimensionless ratio

# Temperature anisotropy: Tperp_ham / Tpar_ham of the hammerhead sub-population.
# Hammerheads are PARALLEL-EXTENDED features by construction (they're the
# high-parallel-velocity wings of the proton VDF), so we typically expect
# T_par > T_perp for the hammerhead population itself, giving T_perp/T_par < 1.
# Variations in this ratio track how "fresh" vs "relaxed" the population is.
_tperp_raw = np.asarray(ham.Tperp_ham, dtype=np.float64)
_tpar_raw  = np.asarray(ham.Tpar_ham,  dtype=np.float64)
_safe_tpar = np.where((_tpar_raw > 0) & np.isfinite(_tpar_raw), _tpar_raw, np.nan)
t_aniso_data = _tperp_raw / _safe_tpar   # dimensionless ratio, typically order unity

# Parallel drift of hammerhead relative to core, normalized by core Alfven speed.
# Classic beam-strength parameter. Under the Landau damping picture we expect
# this to correlate POSITIVELY with above-cutoff wave power (more Landau
# pumping -> more pronounced parallel beam -> larger drift).
#
# NOTE: Jaye's reference notebook (Hamstrings_E27/vdrift_commands.ipynb) has
# a typo in compute_vpar where Bz_inst is passed as By_inst. Fixed here.
_vx_c = np.asarray(ham.vx_inst_core, dtype=np.float64)
_vy_c = np.asarray(ham.vy_inst_core, dtype=np.float64)
_vz_c = np.asarray(ham.vz_inst_core, dtype=np.float64)
_vx_h = np.asarray(ham.vx_inst_ham,  dtype=np.float64)
_vy_h = np.asarray(ham.vy_inst_ham,  dtype=np.float64)
_vz_h = np.asarray(ham.vz_inst_ham,  dtype=np.float64)
_Bx   = np.asarray(ham.Bx_inst,      dtype=np.float64)
_By   = np.asarray(ham.By_inst,      dtype=np.float64)
_Bz   = np.asarray(ham.Bz_inst,      dtype=np.float64)

_Bmag_inst = np.sqrt(_Bx**2 + _By**2 + _Bz**2)
_safe_B = np.where((_Bmag_inst > 0) & np.isfinite(_Bmag_inst), _Bmag_inst, np.nan)
_bhat_x = _Bx / _safe_B
_bhat_y = _By / _safe_B
_bhat_z = _Bz / _safe_B

_vpar_core = _vx_c * _bhat_x + _vy_c * _bhat_y + _vz_c * _bhat_z
_vpar_ham  = _vx_h * _bhat_x + _vy_h * _bhat_y + _vz_h * _bhat_z
_vdrift    = _vpar_ham - _vpar_core   # km/s

# Core Alfven speed (21.8 converts nT & cm^-3 to km/s for the proton Alfven speed)
_safe_ncore_pos = np.where((_n_core_raw > 0) & np.isfinite(_n_core_raw), _n_core_raw, np.nan)
_vA_core = 21.8 * _Bmag_inst / np.sqrt(_safe_ncore_pos)   # km/s

# Normalized drift, absolute value (dimensionless beam strength)
_vdrift_over_vA = _vdrift / _vA_core
vdrift_data = np.abs(_vdrift_over_vA)

# Raw absolute parallel drift (km/s, no v_A normalization). Tests whether
# the v_A normalization is what's hiding the signal vs whether the beam
# velocity is genuinely flat with wave power.
vdrift_raw_data = np.abs(_vdrift)  # km/s
# Note: vdrift_data gets NaN-guarded below in the same block that handles
# n_ham_data and t_aniso_data (_nan_interp is defined there, not yet).

# ---- NaN guard for rolling-mean smoothers ----
# scipy.ndimage.uniform_filter1d uses a cumulative-sum internally. A single
# NaN in the input will poison EVERY output position from that index to the
# end of the array (the running sum becomes NaN and stays NaN). To protect
# downstream smoothing, we linearly interpolate across NaNs now. With only
# a handful of NaN samples (n_core or Tpar crossing zero), linear interp
# is a safe fill.
def _nan_interp(y):
    mask = np.isnan(y)
    if not mask.any():
        return y
    if mask.all():
        return y  # nothing to interpolate from
    idx = np.arange(len(y))
    y_out = y.copy()
    y_out[mask] = np.interp(idx[mask], idx[~mask], y[~mask])
    return y_out

_n_n_nans_before = int(np.isnan(n_ham_data).sum())
_n_t_nans_before = int(np.isnan(t_aniso_data).sum())
_n_v_nans_before  = int(np.isnan(vdrift_data).sum())
_n_vr_nans_before = int(np.isnan(vdrift_raw_data).sum())
n_ham_data      = _nan_interp(n_ham_data)
t_aniso_data    = _nan_interp(t_aniso_data)
vdrift_data     = _nan_interp(vdrift_data)
vdrift_raw_data = _nan_interp(vdrift_raw_data)
if (_n_n_nans_before or _n_t_nans_before or _n_v_nans_before or _n_vr_nans_before):
    print(f'  NaN guard: filled {_n_n_nans_before} NaN in n_ham/n_core, '
          f'{_n_t_nans_before} in Tperp/Tpar, '
          f'{_n_v_nans_before} in |vdrift/vA|, '
          f'{_n_vr_nans_before} in |vdrift| raw (via linear interp)')

# --- Compute hamogram_30s (Jaye's "hammogram" -- detection rate binned at 30s) ---
# Hamogram bin width comes from the config at the top of Step 1.
# _bin_hamogram resolves to either bin_sec (single-bin mode) or bin_sec_hamogram (override mode).
bin_sec = _bin_hamogram  # kept as a simple local alias so existing references still work
ham_times_ns = ham_times.astype('datetime64[ns]').astype('int64')
bin_ns = bin_sec * int(1e9)
bin_edges_ns = np.arange(ham_times_ns[0], ham_times_ns[-1] + bin_ns, bin_ns)
hamogram_counts, _ = np.histogram(ham_times_ns, bins=bin_edges_ns)
bin_centers_ns = (bin_edges_ns[:-1] + bin_edges_ns[1:]) // 2
hamogram_times = bin_centers_ns.astype('datetime64[ns]')
print('\n' + '='*78)
print('STEP 3: Overview + compute hamogram_30s')
print('  Measuring: Binning hammerhead detections into 30s windows; visual inspection of all signals')
print('='*78)
print(f'hamogram_{_bin_hamogram}s: {len(hamogram_counts)} bins, total={hamogram_counts.sum()} detections')

# ---- Binning helper for per-detection ham quantities ----
# Given per-detection values y at timestamps times_ns, average the values
# inside each window of size bin_size_sec. Returns (centers_dt, binned, edges).
def _bin_per_detection(values, times_ns, bin_size_sec):
    bn = bin_size_sec * int(1e9)
    edges = np.arange(times_ns[0], times_ns[-1] + bn, bn)
    which = np.digitize(times_ns, edges) - 1  # 0-indexed bin
    n_bins = len(edges) - 1
    valid = (which >= 0) & (which < n_bins)
    out = np.full(n_bins, np.nan)
    for b in range(n_bins):
        mask = valid & (which == b) & np.isfinite(values)
        if mask.any():
            out[b] = values[mask].mean()
    centers_ns = (edges[:-1] + edges[1:]) // 2
    return centers_ns.astype('datetime64[ns]'), out, edges

# Bin n_ham/n_core and Tperp/Tpar into their configured windows.
# In single-bin mode all three targets land on the SAME time grid.
n_ham_binned_times,   n_ham_binned,   _n_ham_edges   = _bin_per_detection(n_ham_data,   ham_times_ns, _bin_n_ham)
t_aniso_binned_times, t_aniso_binned, _t_aniso_edges = _bin_per_detection(t_aniso_data, ham_times_ns, _bin_t_aniso)
# vdrift shares the same grid as t_aniso in single-bin mode (both per-detection, same config)
vdrift_binned_times,  vdrift_binned,  _vdrift_edges  = _bin_per_detection(vdrift_data,     ham_times_ns, _bin_t_aniso)
vdrift_raw_binned_times, vdrift_raw_binned, _vdrift_raw_edges = _bin_per_detection(vdrift_raw_data, ham_times_ns, _bin_t_aniso)

# Fill empty bins (bins that happened to contain zero valid samples) by linear
# interpolation from neighbors. Small gaps only -- our cadence is such that
# every 120s window usually has many detections; the rare empty bin comes from
# quiet periods with no hammerhead detections at all.
def _fill_binned_nans(y):
    mask = np.isnan(y)
    if not mask.any() or mask.all():
        return y
    idx = np.arange(len(y))
    y_out = y.copy()
    y_out[mask] = np.interp(idx[mask], idx[~mask], y[~mask])
    return y_out

_n_ham_nan_before  = int(np.isnan(n_ham_binned).sum())
_t_aniso_nan_before = int(np.isnan(t_aniso_binned).sum())
_vdrift_nan_before     = int(np.isnan(vdrift_binned).sum())
_vdrift_raw_nan_before = int(np.isnan(vdrift_raw_binned).sum())
n_ham_binned      = _fill_binned_nans(n_ham_binned)
t_aniso_binned    = _fill_binned_nans(t_aniso_binned)
vdrift_binned     = _fill_binned_nans(vdrift_binned)
vdrift_raw_binned = _fill_binned_nans(vdrift_raw_binned)

print(f'n_ham/n_core      binned ({_bin_n_ham}s): {len(n_ham_binned)} bins '
      f'({_n_ham_nan_before} empty bins filled)')
print(f'Tperp/Tpar        binned ({_bin_t_aniso}s): {len(t_aniso_binned)} bins '
      f'({_t_aniso_nan_before} empty bins filled)')
print(f'|v_drift/v_A|     binned ({_bin_t_aniso}s): {len(vdrift_binned)} bins '
      f'({_vdrift_nan_before} empty bins filled)')
print(f'|v_drift| raw     binned ({_bin_t_aniso}s): {len(vdrift_raw_binned)} bins '
      f'({_vdrift_raw_nan_before} empty bins filled) -- km/s')

# --- 7-panel overview ---
fig, axes = plt.subplots(7, 1, figsize=(14, 18), sharex=True)

# Panel 1: B_N time series (downsampled for plotting)
ax1 = axes[0]
skip = 1000
ax1.plot(mag_times[::skip], bn_data[::skip], color='dodgerblue', linewidth=0.4)
ax1.set_ylabel('$B_N$ (nT)')
ax1.set_title('E27 -- 2026/03/11 07:10-08:30')

# Panel 2: Spectrogram (fast nearest-neighbor shading + rasterized)
ax2 = axes[1]
im = ax2.pcolormesh(t_spec_dt, f, Sxx_db,
                     shading='nearest', cmap='inferno', vmin=-20, rasterized=True)
ax2.set_ylabel('Frequency (Hz)')
ax2.set_yscale('log')
ax2.set_ylim(max(f[1], 0.01), fs/2)

# Panel 3: n_ham (physical density, log scale)
ax3 = axes[2]
ax3.plot(ham_times, n_ham_data, 'r.-', markersize=2, linewidth=0.5)
ax3.set_ylabel('n_ham / n_core  (fraction)')
ax3.set_yscale('log')
ax3.set_ylim(1e-5, 1)   # n_ham/n_core actual range ~1.7e-5 to 0.97

# Panel 4 (NEW): temperature anisotropy Tperp_ham / Tpar_ham
ax4 = axes[3]
ax4.plot(ham_times, t_aniso_data, 'g.-', markersize=2, linewidth=0.5)
ax4.set_ylabel('T_perp / T_par\n(ham population)')
ax4.set_yscale('log')
ax4.set_ylim(0.1, 10)
ax4.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.5)

# Panel 5: |v_drift / v_A| (normalized beam strength)
ax5 = axes[4]
ax5.plot(ham_times, vdrift_data, color='purple', marker='.', markersize=2, linestyle='-', lw=0.5)
ax5.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.5)
ax5.set_ylabel('|v_drift_hc / v_A|\n(beam strength)', color='purple')
ax5.set_yscale('log')
ax5.set_ylim(0.01, 10)
ax5.tick_params(axis='y', labelcolor='purple')

# Panel 6 (NEW): |v_drift| raw (km/s) -- unnormalized parallel drift magnitude
ax6 = axes[5]
ax6.plot(ham_times, vdrift_raw_data, color='teal', marker='.', markersize=2, linestyle='-', lw=0.5)
ax6.set_ylabel('|v_drift_hc|\n(km/s, raw)', color='teal')
ax6.set_yscale('log')
ax6.tick_params(axis='y', labelcolor='teal')

# Panel 7: hamogram_{bin_sec}s (Jaye's hammogram -- detection rate)
ax7 = axes[6]
ax7.bar(hamogram_times, hamogram_counts, width=np.timedelta64(bin_sec, 's'),
        color='darkorange', edgecolor='none', alpha=0.9)
ax7.set_ylabel(f'Hammerheads\nper {bin_sec}s')
ax7.set_xlabel('Time (UTC)')

# Attach colorbar to ax2 without stealing space; invisible spacers on the others
for ax in (ax1, ax2, ax3, ax4, ax5, ax6, ax7):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='1.5%', pad=0.1)
    if ax is ax2:
        plt.colorbar(im, cax=cax, label='PSD (dB)')
    else:
        cax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================================
# Step 3b: Tune the split point  (THIS IS THE TUNING CELL)
# ==========================================================================
# Divide the spectrum in two at f_cutoff:
#   above cutoff (>= f_cutoff)  -- the target range we think drives hammerhead formation
#   below cutoff  (< f_cutoff)   -- the disjoint control
# Edit f_cutoff below, re-run this cell to see the split overlaid on the
# spectrogram, then re-run Steps 4-8 for the correlation numbers.
f_cutoff = 10    # Hz  (split point -- above cutoff is >= f_cutoff, below cutoff is < f_cutoff)

fig, ax = plt.subplots(figsize=(14, 6))
# shading='nearest' is much faster than 'gouraud' (no per-vertex interpolation)
im = ax.pcolormesh(t_spec_dt, f, Sxx_db,
                    shading='nearest', cmap='inferno', vmin=-20, rasterized=True)
ax.set_yscale('log')
ax.set_ylim(max(f[1], 0.01), fs/2)
ax.set_ylabel('Frequency (Hz)')
ax.set_xlabel('Time (UTC)')
ax.set_title(f'$B_N$ Spectrogram -- split point at {f_cutoff} Hz  '
             f'(above cutoff >= {f_cutoff} Hz, below cutoff < {f_cutoff} Hz)')

# Draw the split point
ax.axhline(f_cutoff, color='cyan', linestyle='--', linewidth=2.5,
           label=f'split point = {f_cutoff} Hz')
# Shade the above cutoff region so it's obvious what we're targeting
ax.axhspan(f_cutoff, fs/2, alpha=0.12, color='white')

ax.legend(loc='upper right', framealpha=0.9)
plt.colorbar(im, ax=ax, label='PSD (dB)')
plt.tight_layout()
plt.show()

band_mask_tune = (f >= f_cutoff)
print('\n' + '='*78)
print('STEP 3b: Tune the split point')
print('  Measuring: Choosing f_cutoff, the frequency that divides above cutoff (target) from below cutoff (control)')
print('='*78)
print(f'above cutoff (>= {f_cutoff} Hz): {band_mask_tune.sum()} frequency bins out of {len(f)}')
print(f'   actual edges: {f[band_mask_tune][0]:.4f} - {f[band_mask_tune][-1]:.4f} Hz')
print(f'below cutoff  (< {f_cutoff} Hz):  {(~band_mask_tune & (f > 0)).sum()} frequency bins')
print(f'   actual edges: {f[1]:.4f} - {f[~band_mask_tune & (f > 0)][-1]:.4f} Hz')


In [ ]:
# ==========================================================================
# Step 4: Above-cutoff power vs per-detection ham quantities (natively binned)
# ==========================================================================
# Question: is above-cutoff wave power correlated with
#   (A) n_ham / n_core       -- hammerhead fraction of the proton population
#   (B) Tperp_ham / Tpar_ham -- temperature anisotropy of the ham population
#
# Both targets are now binned into 120-second windows (same treatment as
# hamogram). This gives us ~40 fully independent samples per target -- no
# smoothing, no interpolation, no autocorrelation bookkeeping. Band power
# gets rebinned onto the same 120s grid so everything is apples-to-apples.

# --- Extract band power and below-cutoff control from the spectrogram ---
band_mask  = (f >= f_cutoff)
below_mask = (f > 0) & (f < f_cutoff)
band_power_raw  = Sxx[band_mask,  :].mean(axis=0)
below_power_raw = Sxx[below_mask, :].mean(axis=0)

print('\n' + '='*78)
print('STEP 4: Above-cutoff PSD  vs  n_ham/n_core  AND  Tperp/Tpar  (120s binned)')
print('  Measuring: is above-cutoff wave power correlated with hammerhead fraction')
print('             AND with hammerhead temperature anisotropy? (native binning)')
print('='*78)
print(f'Above cutoff (>={f_cutoff} Hz): {band_mask.sum()} spectrogram frequency bins')
print(f'Below cutoff (<{f_cutoff} Hz):  {below_mask.sum()} spectrogram frequency bins')

# --- Rebin band/below power onto the per-detection ham grid(s) ---
# In single-bin mode, _n_ham_edges == _t_aniso_edges, so one rebin is reused.
_t_spec_ns_int = t_spec_dt.astype('datetime64[ns]').astype('int64')

def _rebin_spec_to_grid(spec_power, edges_ns):
    """Mean PSD within each [edges[i], edges[i+1]) bin."""
    which = np.digitize(_t_spec_ns_int, edges_ns) - 1
    n_bins = len(edges_ns) - 1
    out = np.full(n_bins, np.nan)
    for b in range(n_bins):
        mask = (which == b)
        if mask.any():
            out[b] = spec_power[mask].mean()
    return out

# Rebin for n_ham/n_core target
band_on_nh_grid  = _rebin_spec_to_grid(band_power_raw,  _n_ham_edges)
below_on_nh_grid = _rebin_spec_to_grid(below_power_raw, _n_ham_edges)

# Rebin for Tperp/Tpar target (same grid in single-bin mode, separate if overridden)
if use_single_bin:
    band_on_ta_grid  = band_on_nh_grid
    below_on_ta_grid = below_on_nh_grid
else:
    band_on_ta_grid  = _rebin_spec_to_grid(band_power_raw,  _t_aniso_edges)
    below_on_ta_grid = _rebin_spec_to_grid(below_power_raw, _t_aniso_edges)

# --- Correlation helper (Pearson, log-Pearson, log-log Pearson, Spearman) ---
def corr_quad(x, y):
    m = ~np.isnan(x) & ~np.isnan(y) & (x > 0)
    if m.sum() < 3:
        return None, None, None, None, 0
    rp,  _ = pearsonr(x[m], y[m])
    rpL, _ = pearsonr(np.log10(x[m]), y[m])
    rs,  _ = spearmanr(x[m], y[m])
    rpLL = None
    mLL = m & (y > 0)
    if mLL.sum() >= 3:
        rpLL, _ = pearsonr(np.log10(x[mLL]), np.log10(y[mLL]))
    return rp, rpL, rpLL, rs, int(m.sum())

# --- Correlations for both variables (above-cutoff band) ---
nh_above = corr_quad(band_on_nh_grid,  n_ham_binned)
ta_above = corr_quad(band_on_ta_grid,  t_aniso_binned)
nh_below = corr_quad(below_on_nh_grid, n_ham_binned)
ta_below = corr_quad(below_on_ta_grid, t_aniso_binned)

def _row(label, res):
    rp, rpL, rpLL, rs, n = res
    rpLL_s = f'{rpLL:+.3f}' if rpLL is not None else '  n/a'
    print(f'  {label:<30}  Pearson={rp:+.3f}  log-Pearson={rpL:+.3f}  '
          f'log-log={rpLL_s}  Spearman={rs:+.3f}  (n={n})')

print('\n=== n_ham / n_core  (binned, 120s) ===')
_row('above cutoff', nh_above)
_row('below cutoff (control)', nh_below)
print('\n=== Tperp_ham / Tpar_ham  (binned, 120s) ===')
_row('above cutoff', ta_above)
_row('below cutoff (control)', ta_above if use_single_bin and False else ta_below)

# ============================================================
# Plot set: 3-panel stack (band + both ham variables) on the shared grid
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: above-cutoff band power (binned)
ax = axes[0]
ax.plot(n_ham_binned_times, band_on_nh_grid, color='dodgerblue', lw=2.0,
        label='above cutoff')
ax.plot(n_ham_binned_times, below_on_nh_grid, color='gray', lw=1.5,
        linestyle='--', label='below cutoff (control)')
ax.set_ylabel('PSD (binned)')
ax.set_yscale('log')
ax.legend(loc='upper right', fontsize=11)
ax.set_title('Above/below cutoff power, binned to match ham targets', fontweight='bold')

# Panel 2: n_ham/n_core binned
ax = axes[1]
ax.plot(n_ham_binned_times, n_ham_binned, 'ro-', markersize=5, lw=1.2)
ax.set_ylabel('n_ham / n_core\n(binned)', color='red')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='red')
rp, rpL, rpLL, rs, n = nh_above
rpLL_s = f'{rpLL:+.3f}' if rpLL is not None else 'n/a'
ax.set_title(f'n_ham/n_core  vs  above-cutoff PSD  |  '
             f'Pearson={rp:+.3f}  log-log={rpLL_s}  Spearman={rs:+.3f}  (n={n})',
             fontweight='bold', fontsize=12)

# Panel 3: Tperp/Tpar binned
ax = axes[2]
ax.plot(t_aniso_binned_times, t_aniso_binned, color='forestgreen', marker='o',
        markersize=5, linestyle='-', lw=1.2)
ax.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_ylabel('T_perp / T_par\n(binned)', color='forestgreen')
ax.set_yscale('log')
ax.set_ylim(0.1, 10)
ax.tick_params(axis='y', labelcolor='forestgreen')
ax.set_xlabel('Time (UTC)')
rp, rpL, rpLL, rs, n = ta_above
rpLL_s = f'{rpLL:+.3f}' if rpLL is not None else 'n/a'
ax.set_title(f'Tperp/Tpar  vs  above-cutoff PSD  |  '
             f'Pearson={rp:+.3f}  log-log={rpLL_s}  Spearman={rs:+.3f}  (n={n})',
             fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

# ============================================================
# Scatter plots with power-law fits (2x2: above/below for each variable)
# ============================================================
def _loglog_fit(x, y):
    m = ~np.isnan(x) & ~np.isnan(y) & (x > 0) & (y > 0)
    if m.sum() < 3:
        return None, None, None, None, 0
    logx = np.log10(x[m]); logy = np.log10(y[m])
    slope, intercept = np.polyfit(logx, logy, 1)
    xl = np.logspace(logx.min(), logx.max(), 200)
    yl = 10**intercept * xl**slope
    return x[m], y[m], xl, yl, (slope, intercept)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (0,0) n_ham vs above cutoff
ax = axes[0, 0]
xN, yN, xlN, ylN, fitN = _loglog_fit(band_on_nh_grid, n_ham_binned)
ax.scatter(xN, yN, alpha=0.7, s=40, color='red', edgecolor='darkred', linewidth=0.5)
if xlN is not None:
    ax.plot(xlN, ylN, color='crimson', lw=2,
            label=rf'$y = {10**fitN[1]:.2g}\, x^{{{fitN[0]:.2f}}}$')
    ax.legend(loc='upper left', fontsize=10)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Mag PSD above cutoff (binned)')
ax.set_ylabel('n_ham / n_core (binned)')
rp, rpL, rpLL, rs, n = nh_above
ax.set_title('n_ham/n_core  vs  ABOVE cutoff', fontweight='bold', fontsize=13, pad=22)
ax.text(0.5, 1.01, f'Pearson={rp:+.3f}  log-log={rpLL:+.3f} ' if rpLL is not None else f'Pearson={rp:+.3f}  log-log=n/a' ,
        transform=ax.transAxes, ha='center', va='bottom', fontsize=11)

# (0,1) n_ham vs below cutoff
ax = axes[0, 1]
xN, yN, xlN, ylN, fitN = _loglog_fit(below_on_nh_grid, n_ham_binned)
ax.scatter(xN, yN, alpha=0.7, s=40, color='red', edgecolor='darkred', linewidth=0.5)
if xlN is not None:
    ax.plot(xlN, ylN, color='gray', lw=2,
            label=rf'$y = {10**fitN[1]:.2g}\, x^{{{fitN[0]:.2f}}}$')
    ax.legend(loc='upper left', fontsize=10)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Mag PSD below cutoff (binned, control)')
ax.set_ylabel('n_ham / n_core (binned)')
rp, rpL, rpLL, rs, n = nh_below
ax.set_title('n_ham/n_core  vs  BELOW cutoff (control)', fontweight='bold', fontsize=13, pad=22)
ax.text(0.5, 1.01,
        f'Pearson={rp:+.3f}  log-log={rpLL:+.3f}' if rpLL is not None else f'Pearson={rp:+.3f}  log-log=n/a',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=11)

# (1,0) Tperp/Tpar vs above cutoff
ax = axes[1, 0]
xT, yT, xlT, ylT, fitT = _loglog_fit(band_on_ta_grid, t_aniso_binned)
ax.scatter(xT, yT, alpha=0.7, s=40, color='forestgreen', edgecolor='darkgreen', linewidth=0.5)
if xlT is not None:
    ax.plot(xlT, ylT, color='darkgreen', lw=2,
            label=rf'$y = {10**fitT[1]:.2g}\, x^{{{fitT[0]:.2f}}}$')
    ax.legend(loc='upper left', fontsize=10)
ax.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Mag PSD above cutoff (binned)')
ax.set_ylabel('T_perp / T_par (binned)')
rp, rpL, rpLL, rs, n = ta_above
ax.set_title('Tperp/Tpar  vs  ABOVE cutoff', fontweight='bold', fontsize=13, pad=22)
ax.text(0.5, 1.01,
        f'Pearson={rp:+.3f}  log-log={rpLL:+.3f}' if rpLL is not None else f'Pearson={rp:+.3f}  log-log=n/a',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=11)

# (1,1) Tperp/Tpar vs below cutoff
ax = axes[1, 1]
xT, yT, xlT, ylT, fitT = _loglog_fit(below_on_ta_grid, t_aniso_binned)
ax.scatter(xT, yT, alpha=0.7, s=40, color='forestgreen', edgecolor='darkgreen', linewidth=0.5)
if xlT is not None:
    ax.plot(xlT, ylT, color='gray', lw=2,
            label=rf'$y = {10**fitT[1]:.2g}\, x^{{{fitT[0]:.2f}}}$')
    ax.legend(loc='upper left', fontsize=10)
ax.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.5)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Mag PSD below cutoff (binned, control)')
ax.set_ylabel('T_perp / T_par (binned)')
rp, rpL, rpLL, rs, n = ta_below
ax.set_title('Tperp/Tpar  vs  BELOW cutoff (control)', fontweight='bold', fontsize=13, pad=22)
ax.text(0.5, 1.01,
        f'Pearson={rp:+.3f}  log-log={rpLL:+.3f}' if rpLL is not None else f'Pearson={rp:+.3f}  log-log=n/a',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# Step 5: Above-cutoff power vs hamogram_30s (hammerhead OCCURRENCE RATE)
# ==========================================================================
# Question (THE HEADLINE): is above cutoff wave power correlated with the RATE at
# which hammerheads form? (distinct from "how dense do they get" in Step 4)
# Here we rebin the spectrogram power onto the hamogram's native 30s grid
# so the comparison is apples-to-apples with no interpolation.
# Three metrics: Pearson (linear fit), log-Pearson (power-law fit),
# Spearman (any monotonic shape).

# Smoothing knob for hamogram side.
# Normally 1 (off) because bin_sec is now 120s -- the bins are already
# coarse and independent, so no smoothing is needed. Leave at 1 unless
# you want to explore additional low-passing.
# Shim so legacy corr_triple references still work (Step 4 defines corr_quad now)
if 'corr_quad' in globals():
    def corr_triple(x, y):
        rp, rpL, rpLL, rs, n = corr_quad(x, y)
        return rp, rpL, rs, n
else:
    def corr_triple(x, y):
        m = ~np.isnan(x) & ~np.isnan(y) & (x > 0)
        if m.sum() < 3:
            return None, None, None, 0
        rp, _  = pearsonr(x[m], y[m])
        rpL, _ = pearsonr(np.log10(x[m]), y[m])
        rs, _  = spearmanr(x[m], y[m])
        return rp, rpL, rs, int(m.sum())

smooth_factor_hamogram = 1

# --- Rebin band power onto hamogram's 30s grid ---
# For each hamogram bin, average all spectrogram bins that fall inside it.
t_spec_ns_int = t_spec_dt.astype('datetime64[ns]').astype('int64')
which_bin = np.digitize(t_spec_ns_int, bin_edges_ns) - 1  # 0-indexed
valid_bin = (which_bin >= 0) & (which_bin < len(hamogram_counts))

band_power_30s = np.full(len(hamogram_counts), np.nan)
for b in range(len(hamogram_counts)):
    mask = valid_bin & (which_bin == b)
    if mask.any():
        band_power_30s[b] = band_power_raw[mask].mean()

# Smooth both sides
band_30s_smooth = (uniform_filter1d(band_power_30s, size=smooth_factor_hamogram, mode='nearest')
                   if smooth_factor_hamogram > 1 else band_power_30s)
hamogram_smooth = (uniform_filter1d(hamogram_counts.astype(float), size=smooth_factor_hamogram, mode='nearest')
                   if smooth_factor_hamogram > 1 else hamogram_counts.astype(float))

# --- Correlations (shadowed names -- prefix with 'h' so Step 4 values aren't clobbered) ---
rp_h_raw,  rpL_h_raw,  rs_h_raw,  n_h_raw = corr_triple(band_power_30s,  hamogram_counts.astype(float))
rp_h_sm,   rpL_h_sm,   rs_h_sm,   n_h_sm  = corr_triple(band_30s_smooth, hamogram_smooth)

print('\n' + '='*78)
print('STEP 5: Above-cutoff PSD  vs  hamogram_30s  (hammerhead OCCURRENCE RATE)')
print('  Measuring: is band power correlated with the RATE at which hammerheads form?')
print('='*78)
print(f'\n=== hamogram_{bin_sec}s correlations ===')
print(f'RAW       (n={n_h_raw}):')
print(f'   Pearson (raw)   r={rp_h_raw:.3f}')
print(f'   Pearson (log)   r={rpL_h_raw:.3f}   <-- linear fit in log(PSD) space')
print(f'   Spearman        r={rs_h_raw:.3f}')
print(f'SMOOTHED  (n={n_h_sm}):')
print(f'   Pearson (raw)   r={rp_h_sm:.3f}')
print(f'   Pearson (log)   r={rpL_h_sm:.3f}   <-- linear fit in log(PSD) space')
print(f'   Spearman        r={rs_h_sm:.3f}')
print(f'  hamogram smoothing: {smooth_factor_hamogram}x ({smooth_factor_hamogram*bin_sec}s)')

# --- 4-panel stacked plot ---
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Panel 1: RAW band power (rebinned to 30s)
ax = axes[0]
ax.plot(hamogram_times, band_power_30s, color='dodgerblue', lw=0.8, drawstyle='steps-mid')
ax.set_ylabel(f'PSD above cutoff\n(raw, {bin_sec}s bins)', color='dodgerblue')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='dodgerblue')
ax.set_title(f'RAW  |  Pearson r={rp_h_raw:.3f}   log-Pearson r={rpL_h_raw:.3f}   Spearman r={rs_h_raw:.3f}',
             fontweight='bold')

# Panel 2: RAW hamogram_30s
ax = axes[1]
ax.bar(hamogram_times, hamogram_counts, width=np.timedelta64(bin_sec, 's'),
       color='darkorange', edgecolor='none', alpha=0.8)
ax.set_ylabel(f'Hammerheads\nper {bin_sec}s (raw)', color='darkorange')
ax.tick_params(axis='y', labelcolor='darkorange')

# Panel 3: SMOOTHED band power
ax = axes[2]
ax.plot(hamogram_times, band_30s_smooth, color='dodgerblue', lw=1.8)
ax.set_ylabel(f'PSD above cutoff\n(smooth {smooth_factor_hamogram}x)', color='dodgerblue')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='dodgerblue')
ax.set_title(f'SMOOTHED  |  Pearson r={rp_h_sm:.3f}   log-Pearson r={rpL_h_sm:.3f}   Spearman r={rs_h_sm:.3f}',
             fontweight='bold')

# Panel 4: SMOOTHED hamogram
ax = axes[3]
ax.plot(hamogram_times, hamogram_smooth, color='darkorange', lw=1.8)
ax.set_ylabel(f'Hammerheads\nper {bin_sec}s (smooth {smooth_factor_hamogram}x)', color='darkorange')
ax.tick_params(axis='y', labelcolor='darkorange')
ax.set_xlabel('Time (UTC)')

plt.tight_layout()
plt.show()

# --- Side-by-side scatter: above-cutoff (signal) vs below-cutoff (control) ---
# Both with log-linear trend lines so we can visually compare the signal
# band against the disjoint control. A straight line in log(x) vs y space
# means hamogram ~ a + b * log10(PSD) -- hammerhead rate scales with the
# logarithm of wave power.

# Compute below-cutoff power and rebin onto the same 120s hamogram grid
below_power_raw = Sxx[(f > 0) & (f < f_cutoff), :].mean(axis=0)
below_30s = np.full(len(hamogram_counts), np.nan)
for b in range(len(hamogram_counts)):
    mask = valid_bin & (which_bin == b)
    if mask.any():
        below_30s[b] = below_power_raw[mask].mean()
# Match the same smoothing treatment as the above-cutoff side
below_30s_smooth = (uniform_filter1d(below_30s, size=smooth_factor_hamogram, mode='nearest')
                    if smooth_factor_hamogram > 1 else below_30s)

# Correlations for the below-cutoff control (for the panel title)
_r_below = corr_triple(below_30s_smooth, hamogram_smooth)
_rp_below, _rpL_below, _rs_below, _ = _r_below

def _fit_log_linear(x, y):
    m = (~np.isnan(x) & ~np.isnan(y) & (x > 0))
    xv, yv = x[m], y[m]
    logx = np.log10(xv)
    slope, intercept = np.polyfit(logx, yv, 1)
    xl = np.logspace(logx.min(), logx.max(), 200)
    yl = slope * np.log10(xl) + intercept
    return xv, yv, xl, yl, slope, intercept

# Fit both
xA, yA, xlA, ylA, slopeA, interceptA = _fit_log_linear(band_30s_smooth,  hamogram_smooth)
xB, yB, xlB, ylB, slopeB, interceptB = _fit_log_linear(below_30s_smooth, hamogram_smooth)

# Shared y-range so the two panels are visually comparable
_y_lo = min(np.nanmin(yA), np.nanmin(yB))
_y_hi = max(np.nanmax(yA), np.nanmax(yB))
_y_pad = 0.05 * (_y_hi - _y_lo)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Panel A: above cutoff (the signal)
ax = axes[0]
ax.scatter(xA, yA, alpha=0.6, s=20, color='purple', label='data')
ax.plot(xlA, ylA, color='darkorange', lw=2.5,
        label=rf'fit: $y = {slopeA:.2f}\,\log_{{10}}(x) + {interceptA:.2f}$')
ax.set_xlabel(f'Mag PSD above cutoff (>= {f_cutoff} Hz)')
ax.set_ylabel(f'Hammerheads per {bin_sec}s')
ax.set_xscale('log')
ax.legend(loc='upper left')
ax.set_title('ABOVE cutoff (signal)', fontweight='bold', fontsize=16, pad=24)
ax.text(0.5, 1.01,
        f'Pearson={rp_h_sm:+.3f}   log-Pearson={rpL_h_sm:+.3f}   Spearman={rs_h_sm:+.3f}',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=12)
ax.set_ylim(_y_lo - _y_pad, _y_hi + _y_pad)

# Panel B: below cutoff (the control)
ax = axes[1]
ax.scatter(xB, yB, alpha=0.6, s=20, color='gray', label='data')
ax.plot(xlB, ylB, color='darkorange', lw=2.5,
        label=rf'fit: $y = {slopeB:.2f}\,\log_{{10}}(x) + {interceptB:.2f}$')
ax.set_xlabel(f'Mag PSD below cutoff (< {f_cutoff} Hz)')
ax.set_xscale('log')
ax.legend(loc='upper left')
ax.set_title('BELOW cutoff (control)', fontweight='bold', fontsize=16, pad=24)
ax.text(0.5, 1.01,
        f'Pearson={_rp_below:+.3f}   log-Pearson={_rpL_below:+.3f}   Spearman={_rs_below:+.3f}',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=12)
ax.set_ylim(_y_lo - _y_pad, _y_hi + _y_pad)

plt.tight_layout()
plt.show()
print(f'ABOVE-cutoff fit:  slope = {slopeA:+.3f} hams/{bin_sec}s per decade of PSD,  intercept = {interceptA:+.3f}')
print(f'BELOW-cutoff fit:  slope = {slopeB:+.3f} hams/{bin_sec}s per decade of PSD,  intercept = {interceptB:+.3f}')

In [ ]:
# ==========================================================================
# Step 6: Sanity check -- above cutoff vs disjoint below-cutoff control
# ==========================================================================
# For each ham target (hamogram, n_ham/n_core, Tperp/Tpar) we correlate
# against two disjoint frequency ranges:
#   ABOVE cutoff (>= f_cutoff Hz)  -- the target region
#   BELOW cutoff (< f_cutoff Hz)   -- the control
# All three targets are natively binned at bin_sec=120s so the samples are
# fully independent. The DELTA (above - below) tells us whether the
# above-cutoff band has real discriminating power or if we are just seeing
# broadband magnetic variability.

# Power spectra in each range
_band_mask  = (f >= f_cutoff)
_below_mask = (f > 0) & (f < f_cutoff)
_band_ps  = Sxx[_band_mask,  :].mean(axis=0)
_below_ps = Sxx[_below_mask, :].mean(axis=0)

# Rebin spectrogram power onto the 120s ham grid
_t_spec_ns_int = t_spec_dt.astype('datetime64[ns]').astype('int64')
def _rebin_spec(spec, edges):
    which = np.digitize(_t_spec_ns_int, edges) - 1
    n_bins = len(edges) - 1
    out = np.full(n_bins, np.nan)
    for b in range(n_bins):
        mask = (which == b)
        if mask.any():
            out[b] = spec[mask].mean()
    return out

# Use the hamogram grid as the shared reference (identical to n_ham/t_aniso grids
# in single-bin mode)
_band_binned  = _rebin_spec(_band_ps,  bin_edges_ns)
_below_binned = _rebin_spec(_below_ps, bin_edges_ns)

# Simple correlation helper (returns log-Pearson only for the check table)
def _simple_corr(x, y):
    m = ~np.isnan(x) & ~np.isnan(y) & (x > 0)
    if m.sum() < 3:
        return None, None, None, 0
    rp, _  = pearsonr(x[m], y[m])
    rpL, _ = pearsonr(np.log10(x[m]), y[m])
    rs, _  = spearmanr(x[m], y[m])
    return rp, rpL, rs, int(m.sum())

# Correlations: (above, below) for each of the three targets
(bN_p, bN_Lp, bN_s, _) = _simple_corr(_band_binned,  n_ham_binned)
(fN_p, fN_Lp, fN_s, _) = _simple_corr(_below_binned, n_ham_binned)
(bT_p, bT_Lp, bT_s, _) = _simple_corr(_band_binned,  t_aniso_binned)
(fT_p, fT_Lp, fT_s, _) = _simple_corr(_below_binned, t_aniso_binned)
(bH_p, bH_Lp, bH_s, _) = _simple_corr(_band_binned,  hamogram_counts.astype(float))
(fH_p, fH_Lp, fH_s, _) = _simple_corr(_below_binned, hamogram_counts.astype(float))

print('\n' + '='*78)
print('STEP 6: Sanity check -- above cutoff vs disjoint below-cutoff control')
print('  Measuring: does the above-cutoff band have real discriminating power?')
print('='*78)
print(f'above cutoff: >= {f_cutoff} Hz   |   below cutoff: < {f_cutoff} Hz')
print(f'all targets natively binned to {_bin_hamogram}s, n = {n_ham_binned.size} windows')

def _fmt(x):
    return f'{x:>+7.3f}' if x is not None else '   n/a '

print('\n' + '-'*78)
print(f'{"target":<22}{"source":<18}{"Pearson":>12}{"log-Pearson":>14}{"Spearman":>12}')
print('-'*78)
rows = [
    ('n_ham/n_core', 'above cutoff', bN_p, bN_Lp, bN_s),
    ('n_ham/n_core', 'below cutoff', fN_p, fN_Lp, fN_s),
    ('Tperp/Tpar',   'above cutoff', bT_p, bT_Lp, bT_s),
    ('Tperp/Tpar',   'below cutoff', fT_p, fT_Lp, fT_s),
    ('hamogram',     'above cutoff', bH_p, bH_Lp, bH_s),
    ('hamogram',     'below cutoff', fH_p, fH_Lp, fH_s),
]
for t, s, rp, rpL, rs in rows:
    print(f'{t:<22}{s:<18}{_fmt(rp):>12}{_fmt(rpL):>14}{_fmt(rs):>12}')

print('\n' + '-'*78)
print('DELTA (above - below, log-Pearson)')
print('-'*78)
def _d(a, b):
    return (a - b) if (a is not None and b is not None) else None
print(f'  n_ham/n_core :  {_fmt(_d(bN_Lp, fN_Lp))}')
print(f'  Tperp/Tpar   :  {_fmt(_d(bT_Lp, fT_Lp))}')
print(f'  hamogram     :  {_fmt(_d(bH_Lp, fH_Lp))}')
print('  (positive => above cutoff has real discriminating power)')


In [ ]:
# ==========================================================================
# Step 7: Final diagnostic -- all correlations on the native 120s grid
# ==========================================================================
# All three targets are now binned to bin_sec (default 120s) -- no smoothing.
# Each target gives a small number of fully independent samples (n ~ 40).
# We correlate band power (rebinned to the same grid) against each target,
# report Pearson / log-Pearson / log-log / Spearman with honest p-values.
# The below-cutoff control uses the same grid for a fair comparison.

from scipy.stats import t as _t_dist

# --- Band power and below-cutoff power (same spectrogram, two masks) ---
_band_mask  = (f >= f_cutoff)
_below_mask = (f > 0) & (f < f_cutoff)
_band_raw_power  = Sxx[_band_mask,  :].mean(axis=0)
_below_raw_power = Sxx[_below_mask, :].mean(axis=0)

# --- Rebin both onto each target's grid ---
_t_spec_ns_int = t_spec_dt.astype('datetime64[ns]').astype('int64')

def _rebin(spec, edges):
    which = np.digitize(_t_spec_ns_int, edges) - 1
    n_bins = len(edges) - 1
    out = np.full(n_bins, np.nan)
    for b in range(n_bins):
        mask = (which == b)
        if mask.any():
            out[b] = spec[mask].mean()
    return out

_band_on_hg  = _rebin(_band_raw_power,  bin_edges_ns)
_below_on_hg = _rebin(_below_raw_power, bin_edges_ns)
_band_on_nh  = _rebin(_band_raw_power,  _n_ham_edges)
_below_on_nh = _rebin(_below_raw_power, _n_ham_edges)
if use_single_bin:
    _band_on_ta, _below_on_ta = _band_on_nh, _below_on_nh
else:
    _band_on_ta  = _rebin(_band_raw_power,  _t_aniso_edges)
    _below_on_ta = _rebin(_below_raw_power, _t_aniso_edges)

# --- Correlation helper (Pearson, log-Pearson, log-log Pearson, Spearman) ---
def _corr_quad(x, y):
    m = ~np.isnan(x) & ~np.isnan(y) & (x > 0)
    if m.sum() < 3:
        return None, None, None, None, 0
    rp,  _ = pearsonr(x[m], y[m])
    rpL, _ = pearsonr(np.log10(x[m]), y[m])
    rs,  _ = spearmanr(x[m], y[m])
    rpLL = None
    mLL = m & (y > 0)
    if mLL.sum() >= 3:
        rpLL, _ = pearsonr(np.log10(x[mLL]), np.log10(y[mLL]))
    return rp, rpL, rpLL, rs, int(m.sum())

# --- Correlations for all three targets, above + below cutoff ---
nh_above = _corr_quad(_band_on_nh,  n_ham_binned)
nh_below = _corr_quad(_below_on_nh, n_ham_binned)
ta_above = _corr_quad(_band_on_ta,  t_aniso_binned)
ta_below = _corr_quad(_below_on_ta, t_aniso_binned)
hg_above = _corr_quad(_band_on_hg,  hamogram_counts.astype(float))
hg_below = _corr_quad(_below_on_hg, hamogram_counts.astype(float))

# vdrift lives on the same per-detection grid as n_ham/t_aniso
vd_above  = _corr_quad(_band_on_nh,  vdrift_binned)
vd_below  = _corr_quad(_below_on_nh, vdrift_binned)
# Raw (unnormalized) drift — same grid
vdr_above = _corr_quad(_band_on_nh,  vdrift_raw_binned)
vdr_below = _corr_quad(_below_on_nh, vdrift_raw_binned)

# Sample sizes (same for all in single-bin mode)
n_nh = nh_above[4]
n_ta = ta_above[4]
n_vd  = vd_above[4]
n_vdr = vdr_above[4]
n_hg = hg_above[4]

# Convenience aliases so Step 9 / legacy references keep working
nh_band_raw = nh_above
nh_full_raw = nh_below
ta_band_raw = ta_above
ta_full_raw = ta_below
vd_band_raw  = vd_above
vd_full_raw  = vd_below
vdr_band_raw = vdr_above
vdr_full_raw = vdr_below
hg_band_raw = hg_above
hg_full_raw = hg_below
# For backward compat: "smooth" aliases are identical to "raw" now (no smoothing)
nh_band_sm, nh_full_sm = nh_above, nh_below
ta_band_sm, ta_full_sm = ta_above, ta_below
vd_band_sm, vd_full_sm = vd_above, vd_below
vdr_band_sm, vdr_full_sm = vdr_above, vdr_below
hg_band_sm, hg_full_sm = hg_above, hg_below
n_ham_total = n_nh
n_ta_total  = n_ta
n_vd_total  = n_vd
n_vdr_total = n_vdr
n_hg_total  = n_hg

# --- p-value computation ---
def p_from_r(r, n):
    if r is None or n < 3 or abs(r) >= 1.0:
        return None
    t_stat = r * np.sqrt((n - 2) / max(1 - r*r, 1e-12))
    return 2.0 * (1.0 - _t_dist.cdf(abs(t_stat), df=n - 2))

def fmt_p_sci(p):
    if p is None:     return '    n/a'
    if p < 1e-18:     return '<1e-18 '
    return f'{p:.2e}'

def fmt_p_raw(p):
    if p is None:     return '               n/a'
    if p >= 0.001:    return f'{p:.6f}'
    if p >= 1e-9:     return f'{p:.10f}'
    if p >= 1e-15:    return f'{p:.16f}'
    if p >= 1e-18:    return f'{p:.20f}'
    return f'{p:.2e}'

def fmt(x):
    return f'{x:>+7.3f}' if x is not None else '   n/a '

def print_block(title, above, below, n_total):
    print(f'\n{title}')
    print(f'  (n = {n_total}, fully independent, native {_bin_hamogram}s bins)')
    print(f'  {"":<22}{"Pearson":>10}{"log-P":>10}{"log-log":>10}{"Spearman":>11}'
          f'{"p log-P":>12}{"p log-log":>12}{"p Spear":>12}')
    print(f'  {"-"*99}')
    for label, res in [('above cutoff', above), ('below cutoff', below)]:
        rp, rpL, rpLL, rs, n_samp = res
        p_log = p_from_r(rpL,  n_samp)
        p_ll  = p_from_r(rpLL, n_samp) if rpLL is not None else None
        p_spe = p_from_r(rs,   n_samp)
        print(
            f'  {label:<22}'
            f'{fmt(rp):>10}{fmt(rpL):>10}{fmt(rpLL):>10}{fmt(rs):>11}'
            f'{fmt_p_sci(p_log):>12}{fmt_p_sci(p_ll):>12}{fmt_p_sci(p_spe):>12}'
        )

# ============================================================
# PRINT THE TABLE
# ============================================================
print('\n' + '='*110)
print('STEP 7: Final diagnostic -- all correlations, all metrics, all p-values')
print('  Measuring: above-vs-below-cutoff correlations for all three ham targets, native 120s binning')
print('='*110)
print(f'trange: {trange[0]} to {trange[1]}')
print(f'above cutoff: >= {f_cutoff} Hz   |   below cutoff: < {f_cutoff} Hz')
print(f'bin_sec: {_bin_hamogram}s   (use_single_bin={use_single_bin})')
print('-'*110)

print_block('TARGET: n_ham / n_core  (hammerhead fraction)', nh_above, nh_below, n_nh)
print_block('TARGET: Tperp_ham / Tpar_ham  (anisotropy)',    ta_above, ta_below, n_ta)
print_block('TARGET: |v_drift_hc / v_A|  (beam strength, normalized)', vd_above,  vd_below,  n_vd)
print_block('TARGET: |v_drift_hc|  (raw drift, km/s)',              vdr_above, vdr_below, n_vdr)
print_block('TARGET: hamogram  (occurrence rate)',                   hg_above,  hg_below,  n_hg)

# ============================================================
# DELTA block -- does the above-cutoff band do real work?
# ============================================================
print('\n' + '='*78)
print('DELTA (above - below cutoff)')
print('='*78)
def _dll(ba, fa):
    return (ba - fa) if (ba is not None and fa is not None) else None
def _df(v):
    return f'{v:>+7.3f}' if v is not None else '   n/a '

print(f'{"":<22}{"log-Pearson":>14}{"log-log":>14}{"Spearman":>14}')
print(f'{"-"*66}')
for lbl, ab, be in [
    ('n_ham/n_core',  nh_above,  nh_below),
    ('Tperp/Tpar',    ta_above,  ta_below),
    ('|vdrift/vA|',   vd_above,  vd_below),
    ('|vdrift| raw',  vdr_above, vdr_below),
    ('hamogram',      hg_above,  hg_below),
]:
    dLP  = _dll(ab[1], be[1])
    dLL  = _dll(ab[2], be[2])
    dSp  = _dll(ab[3], be[3])
    print(f'  {lbl:<20}{_df(dLP):>14}{_df(dLL):>14}{_df(dSp):>14}')
print('  (positive => above cutoff has real discriminating power over the control)')

# ============================================================
# SIGNIFICANT CORRELATIONS IDENTIFIED (|r| >= 0.5)
# ============================================================
THRESHOLD = 0.5
all_rows = []
for target_name, ab, be, n_use in [
    ('n_ham/n_core', nh_above,  nh_below,  n_nh),
    ('Tperp/Tpar',   ta_above,  ta_below,  n_ta),
    ('|vdrift/vA|',  vd_above,  vd_below,  n_vd),
    ('|vdrift| raw', vdr_above, vdr_below, n_vdr),
    ('hamogram',     hg_above,  hg_below,  n_hg),
]:
    for source_name, row in [('above cutoff', ab), ('below cutoff', be)]:
        rp, rpL, rpLL, rs, _ = row
        for mname, r_val in [
            ('Pearson',     rp),
            ('log-Pearson', rpL),
            ('log-log',     rpLL),
            ('Spearman',    rs),
        ]:
            if r_val is not None and abs(r_val) >= THRESHOLD:
                all_rows.append((target_name, source_name, mname, r_val, n_use))

all_rows.sort(key=lambda r: abs(r[3]), reverse=True)

print('\n' + '='*110)
print(f'SIGNIFICANT CORRELATIONS IDENTIFIED  (|r| >= {THRESHOLD})')
print('='*110)
if not all_rows:
    print(f'  No correlations above |r| = {THRESHOLD}.')
else:
    print(f'  {"rank":>4}  {"target":<16}{"source":<18}{"metric":<14}{"r":>9}'
          f'{"p (sci)":>12}{"p (raw)":>26}')
    print('  ' + '-'*98)
    for rank, (target, source, metric, r_val, n_use) in enumerate(all_rows, 1):
        p = p_from_r(r_val, n_use)
        print(f'  {rank:>4}  {target:<16}{source:<18}{metric:<14}'
              f'{r_val:>+9.3f}{fmt_p_sci(p):>12}{fmt_p_raw(p):>26}')
print('='*110)


In [ ]:
# ==========================================================================
# Step 8: Visual confirmation (30s-binned overlay)
# ==========================================================================
# If the correlation is real, the above cutoff curve should visibly dance with
# the hamogram bars while the below cutoff curve meanders independently. Same
# 30s grid as Step 7 -- no new analysis, just eyeball verification.

print('\n' + '='*78)
print('STEP 8: Visual confirmation (plots only)')
print('  Measuring: nothing new -- same 30s-binned quantities from Step 7,')
print('             overlaid so you can eyeball whether the correlation numbers')
print('             match what your eye sees.')
print('='*78)

# Rebin both power time series onto the 30s hamogram grid
_band_mask_9 = (f >= f_cutoff)
_band_raw_9  = Sxx[_band_mask_9, :].mean(axis=0)
_full_raw_9  = Sxx[(f > 0) & (f < f_cutoff), :].mean(axis=0)

def _rebin_30s_9(y):
    out = np.full(len(hamogram_counts), np.nan)
    for b in range(len(hamogram_counts)):
        m = valid_bin & (which_bin == b)
        if m.any():
            out[b] = y[m].mean()
    return out

band_30s_raw = _rebin_30s_9(_band_raw_9)
full_30s_raw = _rebin_30s_9(_full_raw_9)

# Optional smoothing on top of the 30s bins (set to 1 to see the raw 30s signal)
_smooth = smooth_factor_hamogram  # reuse the Step 5 knob
if _smooth > 1:
    band_30s_plot = uniform_filter1d(band_30s_raw, size=_smooth, mode='nearest')
    full_30s_plot = uniform_filter1d(full_30s_raw, size=_smooth, mode='nearest')
    ham_plot      = uniform_filter1d(hamogram_counts.astype(float), size=_smooth, mode='nearest')
    _label_tag    = f'smoothed {_smooth}x (~{_smooth*bin_sec}s)'
else:
    band_30s_plot = band_30s_raw
    full_30s_plot = full_30s_raw
    ham_plot      = hamogram_counts.astype(float)
    _label_tag    = f'raw {bin_sec}s bins'

# --- Dual-axis overlay: both power curves on log-y left, hamogram on right ---
fig, ax_left = plt.subplots(figsize=(14, 6))

ax_left.plot(hamogram_times, band_30s_plot, color='dodgerblue', lw=2.0,
             label=f'above cutoff (>= {f_cutoff} Hz)')
ax_left.plot(hamogram_times, full_30s_plot, color='gray', lw=2.0,
             label='below cutoff', linestyle='--')
ax_left.set_ylabel(f'PSD ({_label_tag})')
ax_left.set_yscale('log')
ax_left.legend(loc='upper left')
ax_left.set_xlabel('Time (UTC)')

ax_right = ax_left.twinx()
ax_right.bar(hamogram_times, ham_plot, width=np.timedelta64(bin_sec, 's'),
             color='darkorange', edgecolor='none', alpha=0.4,
             label=f'hamogram ({_label_tag})')
ax_right.set_ylabel(f'Hammerheads per {bin_sec}s', color='darkorange')
ax_right.tick_params(axis='y', labelcolor='darkorange')
ax_right.legend(loc='upper right')

ax_left.set_title(
    f'30s-binned overlay  |  band vs below cutoff vs hamogram  ({_label_tag})',
    fontweight='bold'
)
plt.tight_layout()
plt.show()

# --- Stacked version: below cutoff on top, above cutoff in middle (adjacent to
# hamogram for direct visual comparison), hamogram on bottom ---
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1 (top): BELOW CUTOFF (control, should NOT dance with hamogram)
ax = axes[0]
ax.plot(hamogram_times, full_30s_plot, color='gray', lw=1.8)
ax.set_ylabel(f'PSD < {f_cutoff} Hz\n({_label_tag})', color='gray')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='gray')
ax.set_title(f'Below cutoff (< {f_cutoff} Hz), binned to {bin_sec}s', fontweight='bold')

# Panel 2 (middle): ABOVE CUTOFF (the target, should track the hamogram below)
ax = axes[1]
ax.plot(hamogram_times, band_30s_plot, color='dodgerblue', lw=1.8)
ax.set_ylabel(f'PSD >= {f_cutoff} Hz\n({_label_tag})', color='dodgerblue')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='dodgerblue')
ax.set_title(f'Above cutoff (>= {f_cutoff} Hz), binned to {bin_sec}s', fontweight='bold')

# Panel 3 (bottom): HAMOGRAM -- right next to the above cutoff so your eye can
# trace the relationship directly across the shared boundary
ax = axes[2]
ax.bar(hamogram_times, ham_plot, width=np.timedelta64(bin_sec, 's'),
       color='darkorange', edgecolor='none', alpha=0.9)
ax.set_ylabel(f'Hammerheads\nper {bin_sec}s')
ax.set_xlabel('Time (UTC)')
ax.set_title(f'hamogram_{bin_sec}s ({_label_tag})', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================================
# Step 9: Findings summary (paper-ready, dynamic)
# ==========================================================================
# Plain-English readout of every correlation computed in Step 7, populated
# with the live numbers. All three targets are on the native 120s grid --
# no smoothing, no effective-n bookkeeping, scipy p-values are the honest
# p-values.

# Unpack the six result tuples from Step 7
_nh_rp, _nh_rpL, _nh_rpLL, _nh_rs, _ = nh_above
_ta_rp, _ta_rpL, _ta_rpLL, _ta_rs, _ = ta_above
_hg_rp, _hg_rpL, _hg_rpLL, _hg_rs, _ = hg_above
_vd_rp, _vd_rpL, _vd_rpLL, _vd_rs, _ = vd_above
_vdr_rp, _vdr_rpL, _vdr_rpLL, _vdr_rs, _ = vdr_above
_nh_lo_rp, _nh_lo_rpL, _nh_lo_rpLL, _nh_lo_rs, _ = nh_below
_ta_lo_rp, _ta_lo_rpL, _ta_lo_rpLL, _ta_lo_rs, _ = ta_below
_vd_lo_rp, _vd_lo_rpL, _vd_lo_rpLL, _vd_lo_rs, _ = vd_below
_vdr_lo_rp, _vdr_lo_rpL, _vdr_lo_rpLL, _vdr_lo_rs, _ = vdr_below
_hg_lo_rp, _hg_lo_rpL, _hg_lo_rpLL, _hg_lo_rs, _ = hg_below

# p-values (all on honest fully-independent sample counts)
_p_hg_rpL  = p_from_r(_hg_rpL,  n_hg_total)
_p_hg_rs   = p_from_r(_hg_rs,   n_hg_total)
_p_hg_lo_rpL = p_from_r(_hg_lo_rpL, n_hg_total)

_p_nh_rp   = p_from_r(_nh_rp,   n_ham_total)
_p_nh_rpL  = p_from_r(_nh_rpL,  n_ham_total)
_p_nh_rpLL = p_from_r(_nh_rpLL, n_ham_total) if _nh_rpLL is not None else None
_p_nh_rs   = p_from_r(_nh_rs,   n_ham_total)

_p_ta_rp   = p_from_r(_ta_rp,   n_ta_total)
_p_ta_rpL  = p_from_r(_ta_rpL,  n_ta_total)
_p_ta_rpLL = p_from_r(_ta_rpLL, n_ta_total) if _ta_rpLL is not None else None
_p_ta_rs   = p_from_r(_ta_rs,   n_ta_total)

_p_vd_rp   = p_from_r(_vd_rp,   n_vd_total)
_p_vd_rpL  = p_from_r(_vd_rpL,  n_vd_total)
_p_vd_rpLL = p_from_r(_vd_rpLL, n_vd_total) if _vd_rpLL is not None else None
_p_vd_rs   = p_from_r(_vd_rs,   n_vd_total)

_p_vdr_rp   = p_from_r(_vdr_rp,   n_vdr_total)
_p_vdr_rpL  = p_from_r(_vdr_rpL,  n_vdr_total)
_p_vdr_rpLL = p_from_r(_vdr_rpLL, n_vdr_total) if _vdr_rpLL is not None else None
_p_vdr_rs   = p_from_r(_vdr_rs,   n_vdr_total)

def _fmt_p_inline(p):
    if p is None:   return 'n/a'
    if p < 1e-15:   return '< 1e-15'
    if p < 1e-3:    return f'{p:.2e}'
    return f'{p:.3f}'

def _fmt_r(r):
    return f'{r:+.3f}' if r is not None else '  n/a '

# Deltas (above - below, log-Pearson)
_delta_hg = _hg_rpL - _hg_lo_rpL if (_hg_rpL is not None and _hg_lo_rpL is not None) else None

print('\n' + '='*78)
print('STEP 9: Findings summary (paper-ready, dynamic)')
print('  Measuring: plain-English readout of all Step 7 correlations')
print('='*78)

print()
print(f'  TIME WINDOW : {trange[0]}  to  {trange[1]}')
print(f'  SPLIT POINT : {f_cutoff} Hz')
print(f'  BIN SIZE    : {_bin_hamogram}s  (use_single_bin={use_single_bin})')
print(f'  n (all three targets, fully independent): {n_hg_total}')
print()

# ========================================================================
# HEADLINE
# ========================================================================
print('-'*78)
print('HEADLINE')
print('-'*78)
print(f'Hammerhead occurrence rate correlates with above-cutoff (>= {f_cutoff} Hz)')
print(f'wave power in B_N. Natively binned at {_bin_hamogram}s, no smoothing:')
print()
print(f'    log-Pearson r = {_fmt_r(_hg_rpL)}    p = {_fmt_p_inline(_p_hg_rpL)}')
print(f'    Spearman    r = {_fmt_r(_hg_rs)}    p = {_fmt_p_inline(_p_hg_rs)}')
print(f'    n = {n_hg_total}  (fully independent samples)')
print()

# ========================================================================
# BAND SPECIFICITY
# ========================================================================
print('-'*78)
print('BAND SPECIFICITY  (the disjoint control)')
print('-'*78)
print(f'The disjoint below-cutoff (< {f_cutoff} Hz) control shows no significant')
print(f'correlation with hammerhead occurrence rate:')
print()
print(f'    log-Pearson r = {_fmt_r(_hg_lo_rpL)}    p = {_fmt_p_inline(_p_hg_lo_rpL)}')
print(f'    Spearman    r = {_fmt_r(_hg_lo_rs)}')
print()
_delta_str = f'{_delta_hg:+.3f}' if _delta_hg is not None else 'n/a'
print(f'Delta (above - below, log-Pearson) = {_delta_str}')
print(f'==> The correlation is specific to the >= {f_cutoff} Hz range.')
print(f'    Broadband magnetic variability does NOT drive this relationship.')
print()

# ========================================================================
# SECONDARY A: n_ham / n_core
# ========================================================================
print('-'*78)
print('SECONDARY A: hammerhead FRACTION (n_ham / n_core)')
print('-'*78)
print(f'Second rigorous finding: the fraction of the proton population classified')
print(f'as hammerhead, averaged over each {_bin_hamogram}s window. Now nearly as strong')
print(f'as the hamogram result -- native binning revealed a signal that was being')
print(f'drowned in per-detection noise.')
print()
print(f'    Pearson       r = {_fmt_r(_nh_rp)}    p = {_fmt_p_inline(_p_nh_rp)}')
print(f'    log-Pearson   r = {_fmt_r(_nh_rpL)}   p = {_fmt_p_inline(_p_nh_rpL)}')
print(f'    log-log       r = {_fmt_r(_nh_rpLL)}   p = {_fmt_p_inline(_p_nh_rpLL)}')
print(f'    Spearman      r = {_fmt_r(_nh_rs)}    p = {_fmt_p_inline(_p_nh_rs)}')
print(f'    n = {n_ham_total}  (120s bins, fully independent)')
print()

# ========================================================================
# SECONDARY B: Tperp / Tpar
# ========================================================================
print('-'*78)
print('SECONDARY B: hammerhead TEMPERATURE ANISOTROPY (Tperp_ham / Tpar_ham)')
print('-'*78)
print(f'Third finding: mean temperature anisotropy of the hammerhead population')
print(f'per {_bin_hamogram}s window. Hammerheads are PARALLEL-extended features by')
print(f'construction, so T_par > T_perp (ratio < 1) for the hammerhead sub-population.')
print(f'Expect a NEGATIVE correlation with wave power if waves drive fresh beam-like')
print(f'hammerheads to form: more waves -> more new parallel-dominated beams ->')
print(f'lower mean T_perp/T_par. The binning refactor revealed this signal clearly.')
print()
print(f'    Pearson       r = {_fmt_r(_ta_rp)}    p = {_fmt_p_inline(_p_ta_rp)}')
print(f'    log-Pearson   r = {_fmt_r(_ta_rpL)}   p = {_fmt_p_inline(_p_ta_rpL)}')
print(f'    log-log       r = {_fmt_r(_ta_rpLL)}   p = {_fmt_p_inline(_p_ta_rpLL)}')
print(f'    Spearman      r = {_fmt_r(_ta_rs)}    p = {_fmt_p_inline(_p_ta_rs)}')
print(f'    n = {n_ta_total}  (120s bins, fully independent)')
print()
print('  Interpretation (Landau damping, per Jaye):')
print('  The correct mechanism is LANDAU DAMPING, not cyclotron scattering.')
print('  Landau resonance (omega - k_par * v_par = 0) acts via the wave\'s')
print('  PARALLEL electric field, so it exchanges energy with v_par only.')
print('  Landau-resonant protons are pumped into the parallel tail of the VDF')
print('  -- which IS the hammerhead by definition. This heats T_par without')
print('  touching T_perp, driving T_perp/T_par DOWN. A single mechanism')
print('  produces all three findings: more waves -> more Landau damping ->')
print('  more beams in the parallel tail (Finding 1, hamogram), more of the')
print('  proton population ends up in the hammerhead state (Finding 2,')
print('  n_ham/n_core), and the population-mean anisotropy is more parallel-')
print('  dominated (Finding 3, T_perp/T_par lower). Three views, one process.')
print()
print('  The band-specific result also has a physical interpretation: the')
print('  cutoff at {} Hz lies near the ion-scale turbulent break (roughly the'.format(f_cutoff))
print('  proton cyclotron frequency for B ~ 500-1000 nT at E27 perihelion).')
print('  Below the break: MHD cascade, no parallel electric fields, no')
print('  Landau channel -> no correlation. Above the break: kinetic range')
print('  with parallel-E field components (kinetic Alfven waves, oblique')
print('  modes) -> Landau damping active -> strong correlation. The split')
print('  point is not empirical tuning, it is the turbulent dissipation')
print('  scale turning on.')
print()

# ========================================================================
# SECONDARY C: |v_drift_hc / v_A|  (beam strength parameter)
# ========================================================================
print('-'*78)
print('SECONDARY C: HAMMERHEAD BEAM STRENGTH (|v_drift_hc / v_A|)')
print('-'*78)
print(f'Fourth finding: parallel drift of the hammerhead population relative to')
print(f'the core, normalized by the core Alfven speed. Classic beam-strength')
print(f'parameter. Under Landau damping there are TWO possible signatures:')
print(f'  (i)  naive prediction: more waves -> bigger drift -> positive correlation')
print(f'  (ii) saturation: the Landau-resonant drift pins at the wave phase')
print(f'       velocity (~v_A for kinetic-range Alfvenic modes). Fraction and')
print(f'       count grow with wave power, but |v_drift/v_A| stays ~1 and shows')
print(f'       NO correlation.')
print()
print(f'    Pearson       r = {_fmt_r(_vd_rp)}    p = {_fmt_p_inline(_p_vd_rp)}')
print(f'    log-Pearson   r = {_fmt_r(_vd_rpL)}   p = {_fmt_p_inline(_p_vd_rpL)}')
print(f'    log-log       r = {_fmt_r(_vd_rpLL)}   p = {_fmt_p_inline(_p_vd_rpLL)}')
print(f'    Spearman      r = {_fmt_r(_vd_rs)}    p = {_fmt_p_inline(_p_vd_rs)}')
print(f'    n = {n_vd_total}  (120s bins, fully independent)')
print()
print('  Interpretation of the null (or near-null) result:')
print('  A strong positive correlation would have been the "simple" confirmation')
print('  of Landau pumping. Its absence does NOT break the Landau damping story,')
print('  but it constrains which version of the story we can tell. If the drift')
print('  is pinned at the wave phase velocity (~v_A for kinetic Alfven modes),')
print('  saturation naturally produces |v_drift/v_A| ~ 1 independent of wave')
print('  amplitude. In that case, the wave ADDS to the fraction and intensity of')
print('  the beam state without shifting its velocity.')
print()

# ---- LANDAU SATURATION DIAGNOSTIC ----
# The "null consistent with saturation" claim is only defensible if
# |v_drift/v_A| is actually clustered near 1.0. If it's scattered all
# over or centered somewhere else, saturation is NOT the right read and
# we need a different explanation for the null.
_vd_arr = np.asarray(vdrift_binned, dtype=np.float64)
_vd_arr = _vd_arr[np.isfinite(_vd_arr)]
if _vd_arr.size > 0:
    _vd_med  = float(np.median(_vd_arr))
    _vd_mean = float(np.mean(_vd_arr))
    _vd_p25  = float(np.percentile(_vd_arr, 25))
    _vd_p75  = float(np.percentile(_vd_arr, 75))
    _vd_min  = float(np.min(_vd_arr))
    _vd_max  = float(np.max(_vd_arr))
    # Measure clustering via the "robust coefficient of variation": IQR / median
    # A CV below ~0.3 means the distribution is tightly clustered around the median
    _vd_rcv = (_vd_p75 - _vd_p25) / max(abs(_vd_med), 1e-9)
    # Saturation velocity (where the drift is pinned)
    _sat_velocity = _vd_med
    # "Tight cluster" threshold: robust CV < 0.3 AND min/max within factor of 3
    _tight_enough = (_vd_rcv < 0.30) and (_vd_max / max(_vd_min, 1e-9) < 3.0)

    print('  ---- Landau saturation diagnostic ----')
    print(f'  |v_drift/v_A| distribution over {_vd_arr.size} binned values:')
    print(f'    median = {_vd_med:.3f}   mean = {_vd_mean:.3f}')
    print(f'    IQR    = [{_vd_p25:.3f}, {_vd_p75:.3f}]   (width = {_vd_p75-_vd_p25:.3f})')
    print(f'    range  = [{_vd_min:.3f}, {_vd_max:.3f}]')
    print(f'    robust coefficient of variation (IQR/median) = {_vd_rcv:.3f}')
    print()
    if _tight_enough:
        print(f'  VERDICT: CONSISTENT WITH SATURATION AT ~{_sat_velocity:.2f} x v_A')
        print(f'    The drift is tightly clustered around v_drift = {_sat_velocity:.2f} * v_A')
        print(f'    (robust CV = {_vd_rcv:.2f}, well below the 0.30 clustering threshold).')
        print(f'    The beam velocity is PINNED, not freely varying -- which is the')
        print(f'    signature of wave-particle saturation at a specific phase velocity.')
        print()
        if 0.9 <= _sat_velocity <= 1.1:
            print(f'    Saturation velocity ~ 1 * v_A: classic Landau resonance with pure')
            print(f'    Alfven waves (phase velocity = v_A).')
        elif 1.3 <= _sat_velocity <= 3.0:
            print(f'    Saturation velocity > 1 * v_A: consistent with Landau resonance')
            print(f'    with KINETIC Alfven waves at finite k_perp. The KAW parallel phase')
            print(f'    velocity is boosted above v_A by the dispersion factor')
            print(f'    sqrt(1 + k_perp^2 * rho_i^2), which at k_perp*rho_i ~ 1-2 naturally')
            print(f'    gives v_phase_par ~ 1.4 - 2.5 * v_A. Our measured {_sat_velocity:.2f} falls in')
            print(f'    this range. The beam velocity pinning therefore points to KAW-type')
            print(f'    modes above the ion break as the dominant Landau channel.')
        elif _sat_velocity > 3.0:
            print(f'    Saturation velocity >> v_A: unusual. Could indicate beam-cyclotron')
            print(f'    or magnetosonic-whistler resonance rather than Alfvenic Landau.')
            print(f'    Worth cross-checking with wavelet polarization analysis.')
        else:
            print(f'    Saturation velocity < v_A: unusual for Landau resonance with')
            print(f'    Alfvenic modes. Could indicate sub-Alfvenic dispersion or a')
            print(f'    different wave population dominating the dissipation.')
    else:
        print('  VERDICT: NOT CLEARLY SATURATED')
        if _vd_rcv >= 0.30:
            print(f'    Robust CV = {_vd_rcv:.2f} is above the 0.30 clustering threshold.')
            print(f'    The drift is not pinned at a specific velocity, so the null')
            print(f'    correlation is unexplained under the saturation picture.')
        if _vd_max / max(_vd_min, 1e-9) >= 3.0:
            print(f'    Range (min={_vd_min:.2f}, max={_vd_max:.2f}) spans more than a factor')
            print(f'    of 3 -- too wide to describe as pinned.')
        print('    Either the Landau picture needs revision, or there is a confound.')
else:
    print('  (Landau saturation diagnostic skipped: no finite vdrift_binned values)')
print()
print('  Findings 1, 2, and 3 remain consistent with the Landau damping story')
print('  regardless of how Finding 4 resolves.')
print()

# ========================================================================
# SECONDARY D: |v_drift_hc| raw (unnormalized, km/s)
# ========================================================================
print('-'*78)
print('SECONDARY D: RAW HAMMERHEAD DRIFT SPEED (|v_drift_hc|, km/s)')
print('-'*78)
print(f'Fifth finding: the same parallel drift as Secondary C but WITHOUT the')
print(f'v_A normalization. Tests whether the v_A normalization is what killed')
print(f'the signal in Finding 4, or whether the beam velocity is genuinely flat')
print(f'regardless of normalization choice.')
print()
print(f'    Pearson       r = {_fmt_r(_vdr_rp)}    p = {_fmt_p_inline(_p_vdr_rp)}')
print(f'    log-Pearson   r = {_fmt_r(_vdr_rpL)}   p = {_fmt_p_inline(_p_vdr_rpL)}')
print(f'    log-log       r = {_fmt_r(_vdr_rpLL)}   p = {_fmt_p_inline(_p_vdr_rpLL)}')
print(f'    Spearman      r = {_fmt_r(_vdr_rs)}    p = {_fmt_p_inline(_p_vdr_rs)}')
print(f'    n = {n_vdr_total}  (120s bins, fully independent)')
print()
print('  Diagnostic: if this is POSITIVE and significant while Finding 4 is null,')
print('  the v_A normalization was masking a real beam-velocity signal (v_A and')
print('  wave power are likely co-varying). If this is ALSO null, the beam')
print('  velocity is genuinely flat with wave power and the Landau-saturation')
print('  picture (drift pinned at the wave phase velocity) is the right read.')
print()

# ========================================================================
# PAPER-READY PARAGRAPH (four-tier mechanism story)
# ========================================================================
print('-'*78)
print('PAPER-READY PARAGRAPH (copy/paste starter)')
print('-'*78)
import textwrap

_date_str = trange[0].split()[0]
_time_range = f"{trange[0].split()[1][:5]}-{trange[1].split()[1][:5]} UT"

_para1 = (
    f'"For the E27 perihelion approach ({_date_str} {_time_range}), '
    f'we find that above-cutoff (>= {f_cutoff} Hz) magnetic power spectral '
    f'density in $B_N$ correlates with four independent hammerhead '
    f'observables at native {_bin_hamogram}-second binning '
    f'(n = {n_hg_total}, fully independent samples). Three show strong '
    f'correlations in the direction predicted by Landau damping of '
    f'kinetic-range fluctuations; the fourth (beam velocity) shows a '
    f'null consistent with Landau saturation.'
)

_para2 = (
    f'The hammerhead occurrence rate shows a strong positive correlation '
    f'(log-Pearson r = {_hg_rpL:.2f}, p = {_fmt_p_inline(_p_hg_rpL)}; '
    f'Spearman r = {_hg_rs:.2f}, p = {_fmt_p_inline(_p_hg_rs)}). '
    f'The hammerhead fraction of the proton population (n_ham / n_core) '
    f'shows a similarly strong positive correlation (Pearson r = {_nh_rp:.2f}, '
    f'p = {_fmt_p_inline(_p_nh_rp)}; Spearman r = {_nh_rs:.2f}, '
    f'p = {_fmt_p_inline(_p_nh_rs)}). '
    f'The hammerhead beam strength, |v_drift_hc / v_A| (parallel drift of the '
    f'hammerhead relative to the core, normalized by the core Alfven speed), '
    f'shows no significant correlation with wave power (Pearson r = {_vd_rp:.2f}, '
    f'p = {_fmt_p_inline(_p_vd_rp)}; Spearman r = {_vd_rs:.2f}, '
    f'p = {_fmt_p_inline(_p_vd_rs)}). However, the drift itself is tightly '
    f'clustered in the binned data (median = {_vd_med:.2f} * v_A, IQR = '
    f'[{_vd_p25:.2f}, {_vd_p75:.2f}], robust coefficient of variation '
    f'= {_vd_rcv:.2f}), indicating that the beam velocity is PINNED at '
    f'roughly {_vd_med:.1f} * v_A rather than freely varying with wave amplitude. '
    f'A pinned drift combined with a null correlation is the signature of '
    f'saturation at a specific wave phase velocity. The measured pinning '
    f'velocity is consistent with Landau-resonant damping of kinetic Alfven '
    f'waves (KAWs) at finite perpendicular wavenumber, for which the parallel '
    f'phase velocity is boosted above v_A by the dispersion factor '
    f'sqrt(1 + k_perp^2 rho_i^2) -- a factor of ~2 at k_perp rho_i ~ 1-2. '
    f'The hammerhead temperature anisotropy (T_perp / T_par) shows a '
    f'significant negative correlation (log-Pearson r = {_ta_rpL:.2f}, '
    f'p = {_fmt_p_inline(_p_ta_rpL)}; Spearman r = {_ta_rs:.2f}, '
    f'p = {_fmt_p_inline(_p_ta_rs)}). All four signatures are physically '
    f'consistent with Landau damping of kinetic-range fluctuations: the '
    f'Landau resonance transfers wave energy into parallel proton motion, '
    f'building the parallel tail that defines the hammerhead population. '
    f'This simultaneously increases the detection rate and fraction, enhances '
    f'the parallel drift (beam strength), and heats T_par without touching '
    f'T_perp (driving T_perp/T_par lower).'
)

_para3 = (
    f'All four analyses are band-specific: a disjoint below-cutoff '
    f'(< {f_cutoff} Hz) control shows no significant correlation with any '
    f'of the four targets (hamogram log-Pearson r = {_hg_lo_rpL:.2f}, '
    f'p = {_fmt_p_inline(_p_hg_lo_rpL)}). The band-specificity is itself '
    f'physically motivated: the {f_cutoff} Hz split point lies near the ion '
    f'cyclotron frequency at E27 perihelion distances, i.e. the turbulent '
    f'break where the MHD cascade gives way to kinetic dispersive modes '
    f'that carry parallel electric-field components and can Landau-resonate '
    f'with protons. Below the break these dynamics are absent (no parallel '
    f'electric field, no Landau damping) and the correlation vanishes."'
)

for para in (_para1, _para2, _para3):
    for line in textwrap.wrap(para, width=76):
        print(f'  {line}')
    print()

print('='*78)


## Notes, caveats, and methodological considerations

This section collects the "why did we do it that way?" material that we intentionally stripped out of the cell comments to keep the analysis flow readable. It's also a pre-emptive answer sheet for the questions a reviewer will likely ask.

### Why smoothing, and what it costs us

Both of our signals (band power and hammerhead parameters) carry fast fluctuations from turbulence and measurement noise. Underneath that jitter, both evolve on slower, physically meaningful timescales — cyclotron wave bursts typically last minutes, and hammerhead populations persist on comparable timescales. The correlation we care about lives in those slow trends. Smoothing is a low-pass filter that suppresses jitter so the trends dominate.

**But smoothing isn't free.** A rolling mean of length $N$ introduces autocorrelation between adjacent output samples — they share $N-1$ of their input values, so they aren't independent anymore. Effectively, $n_{\text{eff}} \approx n / N$. scipy's `pearsonr` / `spearmanr` assume full independence and will report misleadingly small p-values on smoothed data. Our `p_from_r` in Step 7 recomputes p from the t-distribution using the reduced $n_{\text{eff}}$, which is the honest number.

**Practical policy:** Report raw (unsmoothed) correlations as the primary result, use smoothed numbers as "consistent with" secondary support. The raw $n = 158$ hamogram result is fully independent and gives us $p \sim 10^{-13}$ without any corrections.

### Pearson vs log-Pearson vs Spearman — three different questions

| Metric | Tests for | Best when |
|---|---|---|
| Pearson (raw) | Linear relationship $y = a + bx$ | Both variables roughly normal, relationship is linear |
| Pearson (log) | Power-law relationship $y = A\, x^{\alpha}$ | Signal spans orders of magnitude; physics is scaling (common in space plasma) |
| Spearman | Any monotonic relationship (rank-based) | You don't want to assume functional form |

**Diagnostic pattern:** if `log-Pearson > raw-Pearson`, the relationship is nonlinear in the power-law direction (physically motivated). If `Spearman > log-Pearson`, the relationship is monotonic but not cleanly power-law. For hamogram we see the first pattern (clean power law); for n_ham we see a different pattern (Pearson $\approx$ Spearman, log-Pearson lower) suggesting a more linear relationship — but see the next caveat.

### Why extending the above cutoff to Nyquist is fine

Space plasma turbulence follows roughly Kolmogorov scaling (PSD $\sim f^{-5/3}$), so power rolls off hard at high frequencies. Above $\sim$50 Hz there's essentially no power to contribute. That means averaging over `f_cutoff`–Nyquist gives practically the same answer as averaging over `f_cutoff`–50 Hz, with one less knob to tune. We traded an arbitrary upper bound for a cleaner two-way spectrum split.

### The cutoff-tuning caveat (garden of forking paths)

We arrived at the 10 Hz split point by iterating visually: looked at the spectrogram, tried a couple of cutoffs, kept the one with cleaner correlations. That's scientifically fine for **exploration**, but a reviewer will (correctly) flag it as "you tuned the cutoff to maximize the correlation." The way to defend against that:

1. **Freeze the cutoff** and reproduce on an independent encounter (E26, E25...) — gold standard.
2. **Sweep the cutoff** across a reasonable range and show a broad plateau of correlations, not a sharp peak at exactly our chosen value.
3. **Derive it from theory** — e.g. predict the cyclotron frequency at PSP's distance and use that as the cutoff before running on new data.

Any of these converts "we found a cutoff that worked" into "we predicted a cutoff and validated it."

### The n_ham red flag

For n_ham, Pearson jumped from **raw 0.17 to smoothed 0.47** when we added smoothing. That's a much bigger jump than the log-Pearson or the hamogram equivalents. When raw Pearson specifically inflates under smoothing, it often means we're picking up a **shared long-timescale trend** — e.g. both signals drifting together with PSP's trajectory or solar wind regime, rather than fast correlated physics.

**Action item:** detrend both n_ham and the above-cutoff power (subtract a linear fit over the 80-minute window) and recompute. If the n_ham correlation survives, it's a real second finding. If it collapses, it was trajectory contamination and the hamogram story stands alone. This is the single most important follow-up for the secondary finding.

### Effective-n is a conservative estimate

Our $n_{\text{eff}} = n / \text{smoothing\_window}$ is a rough first-order correction. The "true" effective sample size depends on the actual autocorrelation length of the signals, which includes both smoothing-induced and physically-induced autocorrelation. A proper **block bootstrap** (resample in blocks of about the autocorrelation length and rebuild the null distribution) would give us a data-driven p-value that's reviewer-proof. Optional follow-up, not a blocker.

### p-values hitting the floating-point floor

You'll see some Spearman p-values print as `0.00e+00` or `<1e-18`. That's the float precision limit of `scipy.stats.t.cdf` returning exactly 1.0 for large t-statistics, not a genuine zero. The true p-value is around $10^{-18}$ or smaller. For paper writing, "$p < 10^{-15}$" or "below machine precision" is the right phrasing.

### Next steps for paper-grade rigor

1. **Reproduce on a second encounter** with the 10 Hz split point frozen — by far the most important check.
2. **Detrend** both signals, rerun the correlations, confirm the hamogram result survives and see what happens to n_ham.
3. **Block bootstrap** for honest p-values on smoothed data.
4. **Scan the cutoff** across 5–20 Hz and verify the result is on a plateau, not a knife edge.
5. **Derive the expected cutoff from theory** (proton cyclotron frequency at PSP's distance) so the cutoff choice is prediction, not post-hoc tuning.

None of these invalidates the current finding. They're what moves it from "interesting single-encounter result" to "publishable robust finding."
